# 02 — Geocoding Analysis

Analyze the two-layer geocoding system's performance:
- **Match rate**: what percentage of Reddit posts mapped to a county FIPS?
- **Layer breakdown**: static lookup vs GeoPy fallback
- **Unmatched posts**: what locations are we missing aliases for?

This match rate is a KEY metric for the methodology page.

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

In [ ]:
# Load matched and unmatched posts
matched_path = Path('../../data/processed/matched_posts.parquet')
unmatched_path = Path('../../data/processed/unmatched_posts.csv')

if matched_path.exists():
    matched = pd.read_parquet(matched_path)
    print(f'Matched posts: {len(matched)}')
else:
    print('No matched posts file — run preprocessing first')
    matched = pd.DataFrame()

if unmatched_path.exists():
    unmatched = pd.read_csv(unmatched_path)
    print(f'Unmatched posts: {len(unmatched)}')
else:
    unmatched = pd.DataFrame()

In [ ]:
# Match rate calculation
total = len(matched) + len(unmatched)
rate = len(matched) / total * 100 if total > 0 else 0
print(f'\n=== GEOCODING MATCH RATE ===')
print(f'Total posts: {total}')
print(f'Matched: {len(matched)} ({rate:.1f}%)')
print(f'Unmatched: {len(unmatched)} ({100-rate:.1f}%)')
print(f'\nThis number goes on the methodology page.')

In [ ]:
# County distribution of matched posts
if not matched.empty and 'matched_county_fips' in matched.columns:
    county_counts = matched['matched_county_fips'].value_counts()
    fig, ax = plt.subplots(figsize=(12, 5))
    county_counts.head(20).plot.bar(ax=ax, color='#22c55e')
    ax.set_title('Posts per County (top 20)')
    ax.set_ylabel('Post count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# Unmatched locations — candidates for new aliases
if not unmatched.empty and 'raw_locations' in unmatched.columns:
    print('Most common UNMATCHED locations (add these as aliases):')
    # Parse the raw_locations column (stored as string repr of list)
    import ast
    all_locs = []
    for locs in unmatched['raw_locations'].dropna():
        try:
            parsed = ast.literal_eval(locs) if isinstance(locs, str) else locs
            all_locs.extend(parsed)
        except (ValueError, SyntaxError):
            pass
    loc_counts = pd.Series(all_locs).value_counts()
    print(loc_counts.head(20).to_string())

In [ ]:
# Geocode cache stats
cache_path = Path('../../data/reference/geocode_cache.json')
if cache_path.exists():
    with open(cache_path) as f:
        cache = json.load(f)
    hits = sum(1 for v in cache.values() if v is not None)
    misses = sum(1 for v in cache.values() if v is None)
    print(f'Geocode cache: {len(cache)} entries')
    print(f'  Hits (resolved): {hits}')
    print(f'  Misses (None): {misses}')